<a href="https://colab.research.google.com/github/NamishBansal15/substation-detection/blob/main/model-training/yolo_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install roboflow ultralytics wandb -q

import os
import random
import shutil
from collections import defaultdict
from getpass import getpass
from pathlib import Path

import pandas as pd
import wandb
from roboflow import Roboflow
from ultralytics import YOLO
from google.colab import drive

# ── Dataset ─────────────────────────────────────────────────────
ROBOFLOW_API_KEY = getpass("Roboflow API key: ")
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("INSERT PROJECT WORKSPACE").project("INSERT PROJECT NAME")
dataset = project.version(2).download("yolov8")
DATA_YAML = f"{dataset.location}/data.yaml"
print(f"Dataset ready at: {DATA_YAML}")

# ── Drive mount for checkpoints ─────────────────────────────────
drive.mount("/content/drive")

# Change this one path if you want to store checkpoints elsewhere.
DRIVE_DIR = "/content/drive/MyDrive/april-detection/yolo"
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
!pip install wandb roboflow ultralytics -q

import os, shutil, threading, time
from getpass import getpass
from pathlib import Path
from datetime import datetime

import torch
import pandas as pd
from ultralytics import YOLO
import wandb
from roboflow import Roboflow
from google.colab import drive

drive.mount('/content/drive')

# ════════════════════════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════════════════════════
ROBOFLOW_API_KEY = getpass("Roboflow API key: ")
WANDB_PROJECT    = "april-detection-ablation"
ABLATION_BASE_DIR = "/content/drive/MyDrive/april-detection"
RESULTS_FILE     = f"{ABLATION_BASE_DIR}/ablation_results.json"
DRIVE_DIR        = f"{ABLATION_BASE_DIR}/ablation_yolo"

DATASET_TIERS = {
    "augmented": {
        "workspace": "coriell",
        "train":     ("augmented-train-only", 2),
        "valid":     ("augmented-valid",      2),
        "test":      ("augmented-test",       3),
    },
}

CLASSES     = ["Alt Energy", "Circuit Breaker", "Control", "Power Lines", "Reactor", "Transformer"]
NUM_CLASSES = 6
BATCH_SIZES = {"yolov8n": 32, "yolov8s": 16, "yolov8m": 8}

# ════════════════════════════════════════════════════════════════
# HEARTBEAT
# ════════════════════════════════════════════════════════════════
def start_heartbeat():
    def _beat():
        while True:
            print(f"[Heartbeat] {time.strftime('%H:%M:%S')}")
            time.sleep(120)
    threading.Thread(target=_beat, daemon=True).start()

start_heartbeat()

# ════════════════════════════════════════════════════════════════
# DOWNLOAD
# ════════════════════════════════════════════════════════════════
def download_tier(tier_name, tier_cfg):
    rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
    out_dir = f"/content/datasets/{tier_name}"
    os.makedirs(out_dir, exist_ok=True)

    for split in ["train", "valid", "test"]:
        slug, version = tier_cfg[split]
        ds  = rf.workspace(tier_cfg["workspace"]).project(slug).version(version).download("yolov8")
        src = Path(ds.location) / split

        img_dst = Path(out_dir) / split / "images"
        lbl_dst = Path(out_dir) / split / "labels"
        img_dst.mkdir(parents=True, exist_ok=True)
        lbl_dst.mkdir(parents=True, exist_ok=True)

        for f in (src / "images").iterdir():
            if f.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                shutil.copy2(str(f), str(img_dst / f.name))

        for f in (src / "labels").iterdir():
            if f.suffix == ".txt":
                shutil.copy2(str(f), str(lbl_dst / f.name))

        print(f"  [{tier_name}/{split}] "
              f"{len(list(img_dst.glob('*.jpg')))} images, "
              f"{len(list(lbl_dst.glob('*.txt')))} labels")

    with open(f"{out_dir}/data.yaml", "w") as f:
        f.write(f"path: {out_dir}\n"
                f"train: train/images\n"
                f"val: valid/images\n"
                f"test: test/images\n"
                f"nc: {NUM_CLASSES}\n"
                f"names: {CLASSES}\n")

    print(f"✅ Tier '{tier_name}' ready at {out_dir}")
    return out_dir

# ════════════════════════════════════════════════════════════════
# VALIDATE
# ════════════════════════════════════════════════════════════════
def validate_tier(tier_name, data_dir):
    print(f"\n🔍 Validating {tier_name}")
    all_good = True

    for split in ["train", "valid", "test"]:
        img_dir = Path(data_dir) / split / "images"
        lbl_dir = Path(data_dir) / split / "labels"
        n_imgs   = len(list(img_dir.glob("*.jpg"))) + len(list(img_dir.glob("*.png")))
        n_labels = len(list(lbl_dir.glob("*.txt")))
        print(f"  {split}: {n_imgs} images, {n_labels} labels")
        if n_imgs == 0:
            print(f"    ❌ NO IMAGES"); all_good = False
        if n_labels == 0:
            print(f"    ❌ NO LABELS"); all_good = False
        if n_imgs > 0 and abs(n_imgs - n_labels) > n_imgs * 0.05:
            print(f"    ⚠️ image/label count mismatch (>5%)")

    # Check class IDs in a sample of label files
    import random
    labels = list((Path(data_dir) / "train" / "labels").glob("*.txt"))
    sample = random.sample(labels, min(20, len(labels)))
    class_ids = set()
    for lbl in sample:
        with open(lbl) as f:
            for line in f:
                if line.strip():
                    class_ids.add(int(line.split()[0]))
    print(f"  Class IDs in train sample: {sorted(class_ids)} (expected 0-{NUM_CLASSES-1})")
    if max(class_ids) >= NUM_CLASSES:
        print(f"    ❌ CLASS ID OUT OF RANGE"); all_good = False

    if all_good:
        print(f"  ✅ {tier_name} looks good — safe to train")
    else:
        print(f"  🛑 {tier_name} has issues — DO NOT TRAIN")
    return all_good

# ════════════════════════════════════════════════════════════════
# UTILITIES
# ════════════════════════════════════════════════════════════════
def save_metrics(metrics):
    import json
    results = []
    if os.path.exists(RESULTS_FILE):
        with open(RESULTS_FILE) as f:
            results = json.load(f)
    metrics["timestamp"] = datetime.now().isoformat()
    results.append(metrics)
    os.makedirs(Path(RESULTS_FILE).parent, exist_ok=True)
    with open(RESULTS_FILE, "w") as f:
        json.dump(results, f, indent=2)
    print(f"[Metrics] Saved → {metrics['model_name']}")

def sync_checkpoints(src, dst, stop_event):
    os.makedirs(dst, exist_ok=True)
    while not stop_event.is_set():
        try:
            for f in Path(src).rglob("*.pt"):
                d = Path(dst) / f.name
                if not d.exists():
                    shutil.copy2(str(f), str(d))
                    print(f"[Sync] {f.name}")
        except Exception as e:
            print(f"[Sync] Error: {e}")
        time.sleep(300)

# ════════════════════════════════════════════════════════════════
# YOLO
# ════════════════════════════════════════════════════════════════
def train_yolo(model_name, tier_name, data_dir):
    run_name  = f"{model_name}-{tier_name}"
    drive_dir = f"{DRIVE_DIR}/{run_name}"

    print(f"\n🚀 Training {run_name}")

    wandb.init(project=WANDB_PROJECT, name=run_name,
               config={"tier": tier_name, "model": model_name, "epochs": 50},
               reinit=True)

    stop_event  = threading.Event()
    sync_thread = threading.Thread(
        target=sync_checkpoints,
        args=(f"/content/runs/detect/{run_name}/weights", drive_dir, stop_event),
        daemon=True)
    sync_thread.start()

    model = YOLO(f"{model_name}.pt")
    model.train(
        data=f"{data_dir}/data.yaml",
        epochs=50,
        imgsz=640,
        batch=BATCH_SIZES[model_name],
        name=run_name,
        exist_ok=True,
        plots=True,
    )

    stop_event.set()

    # ── Per-epoch metrics from results.csv ───────────────────────
    results_path = f"/content/runs/detect/{run_name}/results.csv"
    if os.path.exists(results_path):
        df = pd.read_csv(results_path)
        df.columns = df.columns.str.strip()
        for _, row in df.iterrows():
            wandb.log({
                "epoch":        int(row["epoch"]),
                "mAP50":        float(row["metrics/mAP50(B)"]),
                "mAP50_95":     float(row["metrics/mAP50-95(B)"]),
                "precision":    float(row["metrics/precision(B)"]),
                "recall":       float(row["metrics/recall(B)"]),
                "box_loss":     float(row["train/box_loss"]),
                "cls_loss":     float(row["train/cls_loss"]),
                "dfl_loss":     float(row["train/dfl_loss"]),
                "val_box_loss": float(row["val/box_loss"]),
                "val_cls_loss": float(row["val/cls_loss"]),
            })
        best = df.loc[df["metrics/mAP50(B)"].idxmax()]
        print(f"  Best mAP50={best['metrics/mAP50(B)']:.4f} at epoch {int(best['epoch'])}")
    else:
        print(f"  ⚠️ results.csv not found")

    # ── Per-class metrics from best weights ──────────────────────
    weights = f"/content/runs/detect/{run_name}/weights/best.pt"
    if os.path.exists(weights):
        val = YOLO(weights).val(data=f"{data_dir}/data.yaml", verbose=False)

        per_class_metrics = {}
        for i, cls in enumerate(CLASSES):
            try:
                per_class_metrics[f"ap50_{cls.lower().replace(' ','_')}"]    = float(val.box.ap50[i])
                per_class_metrics[f"ap50_95_{cls.lower().replace(' ','_')}"] = float(val.box.ap[i])
            except IndexError:
                pass

        save_metrics({
            "model_name":   run_name,
            "tier":         tier_name,
            "architecture": model_name,
            "mAP50":        round(float(val.box.map50), 4),
            "mAP50_95":     round(float(val.box.map),   4),
            "precision":    round(float(val.box.mp),     4),
            "recall":       round(float(val.box.mr),     4),
            "inference_ms": float(val.speed["inference"]),
            "model_size_mb": round(os.path.getsize(weights) / 1e6, 2),
            "per_class":    {k: v for k, v in per_class_metrics.items()
                             if k.startswith("ap50_") and not k.startswith("ap50_95")},
        })

        wandb.log({
            "final_mAP50":     val.box.map50,
            "final_mAP50_95":  val.box.map,
            "final_precision": val.box.mp,
            "final_recall":    val.box.mr,
            "inference_ms":    val.speed["inference"],
            "model_size_mb":   round(os.path.getsize(weights) / 1e6, 2),
            **per_class_metrics
        })

        os.makedirs(drive_dir, exist_ok=True)
        shutil.copy2(weights, f"{drive_dir}/best.pt")
        print(f"  ✅ Best weights saved to Drive")
        print(f"  Inference: {val.speed['inference']:.1f}ms")
    else:
        print(f"  ⚠️ No weights found")

    wandb.finish()
    print(f"✅ Done: {run_name}")

# ════════════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════════════
wandb.login()
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

for tier_name, tier_cfg in DATASET_TIERS.items():
    print(f"\n{'='*60}\nTIER: {tier_name}\n{'='*60}")

    data_dir = download_tier(tier_name, tier_cfg)

    if not validate_tier(tier_name, data_dir):
        print(f"⛔ Skipping {tier_name} — fix dataset first")
        continue

    train_yolo("yolov8n", tier_name, data_dir)
    train_yolo("yolov8s", tier_name, data_dir)
    train_yolo("yolov8m", tier_name, data_dir)

print("\n🎉 Ablation complete!")

In [ ]:


def split_dataset(dataset_location, train_ratio=0.7, val_ratio=0.2, test_ratio=0.1, seed=42):
    random.seed(seed)

    src_images = Path(dataset_location) / "train" / "images"
    src_labels = Path(dataset_location) / "train" / "labels"

    for split in ["valid", "test"]:
        (Path(dataset_location) / split / "images").mkdir(parents=True, exist_ok=True)
        (Path(dataset_location) / split / "labels").mkdir(parents=True, exist_ok=True)

    all_images = list(src_images.glob("*.*"))
    random.shuffle(all_images)

    n = len(all_images)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)

    val_files  = all_images[n_train:n_train + n_val]
    test_files = all_images[n_train + n_val:]

    print(f"Total: {n} → Train: {n_train}, Val: {len(val_files)}, Test: {len(test_files)}")

    val_set  = set(val_files)
    test_set = set(test_files)

    for img_path in val_set | test_set:
        split = "valid" if img_path in val_set else "test"
        label_path = src_labels / (img_path.stem + ".txt")
        shutil.move(str(img_path), Path(dataset_location) / split / "images" / img_path.name)
        if label_path.exists():
            shutil.move(str(label_path), Path(dataset_location) / split / "labels" / label_path.name)

    print("✅ Split complete!")

print("Splitting dataset 70/20/10...")
split_dataset(dataset.location)

DATA_YAML = f"{dataset.location}/data.yaml"

# ── Config ──────────────────────────────────────────────────────
WANDB_PROJECT = "april-detection"
CLASSES = ["Alt Energy", "Circuit Breaker", "Control", "Power Lines", "Reactor", "Transformer"]

configs = [
    {"name": "yolov8n",  "model": "yolov8n.pt",  "epochs": 50, "imgsz": 640, "batch": 32},
    {"name": "yolo11n",  "model": "yolo11n.pt",  "epochs": 50, "imgsz": 640, "batch": 32},
    {"name": "yolo12n",  "model": "yolo12n.pt",  "epochs": 50, "imgsz": 640, "batch": 32},
]

# ── Checkpoint sync ──────────────────────────────────────────────
import threading
import time

def sync_checkpoints(model_name, stop_event):
    src = f"/content/runs/detect/{model_name}/weights"
    dst = f"{DRIVE_DIR}/{model_name}/weights"
    os.makedirs(dst, exist_ok=True)
    while not stop_event.is_set():
        try:
            if os.path.exists(src):
                for f in os.listdir(src):
                    s = f"{src}/{f}"
                    d = f"{dst}/{f}"
                    if not os.path.exists(d):
                        shutil.copy2(s, d)
                        print(f"[Sync] Saved {model_name}/{f} to Drive")
        except Exception as e:
            print(f"[Sync] Error: {e}")
        time.sleep(300)

# ── Training loop ────────────────────────────────────────────────
wandb.login()

for cfg in configs:
    print(f"\n{'='*50}")
    print(f"Starting: {cfg['name']}")
    print(f"{'='*50}")

    wandb.init(
        project=WANDB_PROJECT,
        name=cfg["name"],
        config=cfg,
        reinit=True
    )

    # Start checkpoint sync
    stop_event = threading.Event()
    sync_thread = threading.Thread(
        target=sync_checkpoints,
        args=(cfg["name"], stop_event),
        daemon=True
    )
    sync_thread.start()

    model = YOLO(cfg["model"])

    model.train(
        data=DATA_YAML,
        epochs=cfg["epochs"],
        imgsz=cfg["imgsz"],
        batch=cfg["batch"],
        name=cfg["name"],
        exist_ok=True,
        plots=True,
    )

    # Stop sync
    stop_event.set()

    # ── Per-epoch metrics from results.csv ───────────────────────
    results_path = f"/content/runs/detect/{cfg['name']}/results.csv"
    if os.path.exists(results_path):
        df = pd.read_csv(results_path)
        df.columns = df.columns.str.strip()

        for _, row in df.iterrows():
            wandb.log({
                "epoch":        int(row["epoch"]),
                "mAP50":        float(row["metrics/mAP50(B)"]),
                "mAP50_95":     float(row["metrics/mAP50-95(B)"]),
                "precision":    float(row["metrics/precision(B)"]),
                "recall":       float(row["metrics/recall(B)"]),
                "box_loss":     float(row["train/box_loss"]),
                "cls_loss":     float(row["train/cls_loss"]),
                "dfl_loss":     float(row["train/dfl_loss"]),
                "val_box_loss": float(row["val/box_loss"]),
                "val_cls_loss": float(row["val/cls_loss"]),
            })

        best = df.loc[df["metrics/mAP50(B)"].idxmax()]
        print(f"  Best mAP50={best['metrics/mAP50(B)']:.4f} at epoch {int(best['epoch'])}")
    else:
        print(f"  ⚠️ results.csv not found")

    # ── Per-class metrics from best weights ──────────────────────
    weights = f"/content/runs/detect/{cfg['name']}/weights/best.pt"
    if os.path.exists(weights):
        val_model = YOLO(weights)
        val = val_model.val(data=DATA_YAML, verbose=False)

        per_class_metrics = {}
        for i, cls in enumerate(CLASSES):
            try:
                per_class_metrics[f"ap50_{cls.lower().replace(' ', '_')}"]    = float(val.box.ap50[i])
                per_class_metrics[f"ap50_95_{cls.lower().replace(' ', '_')}"] = float(val.box.ap[i])
            except IndexError:
                pass

        wandb.log({
            "final_mAP50":     float(val.box.map50),
            "final_mAP50_95":  float(val.box.map),
            "final_precision": float(val.box.mp),
            "final_recall":    float(val.box.mr),
            "inference_ms":    float(val.speed["inference"]),
            "model_size_mb":   round(os.path.getsize(weights) / 1e6, 2),
            **per_class_metrics
        })

        # Copy final weights to Drive
        dst_weights = f"{DRIVE_DIR}/{cfg['name']}/weights"
        os.makedirs(dst_weights, exist_ok=True)
        shutil.copy2(weights, f"{dst_weights}/best.pt")
        print(f"  ✅ Best weights saved to Drive")
        print(f"  Inference: {val.speed['inference']:.1f}ms")
    else:
        print(f"  ⚠️ No weights found")

    wandb.finish()
    print(f"✅ Finished: {cfg['name']}")

print("\n🎉 All runs complete!")

In [ ]:
import wandb
from ultralytics import YOLO

api = wandb.Api()
CLASSES = ["Alt Energy", "Circuit Breaker", "Control", "Power Lines", "Reactor", "Transformer"]
models_to_eval = ["yolov8s", "yolov8m", "yolov8l", "yolo11s", "yolo11m", "yolo11l", "yolo12s", "yolo12m"]

if not os.path.exists(DATA_YAML):
    raise FileNotFoundError(
        f"Dataset configuration not found at {DATA_YAML}. "
        "Run the dataset setup/training cells above before evaluating saved checkpoints."
    )

for model_name in models_to_eval:
    weights = f"{DRIVE_DIR}/{model_name}/weights/best.pt"
    if not os.path.exists(weights):
        print(f"⚠️ No weights found for {model_name}")
        continue

    # Resume the wandb run
    runs = api.runs(f"namishemail-george-mason-university/april-detection")
    run_id = next((r.id for r in runs if r.name == model_name), None)
    if not run_id:
        print(f"⚠️ No wandb run found for {model_name}")
        continue

    wandb.init(project="april-detection", name=model_name, id=run_id, resume="must")

    val_model = YOLO(weights)
    val = val_model.val(data=DATA_YAML, verbose=False)

    per_class_metrics = {}
    for i, cls in enumerate(CLASSES):
        try:
            per_class_metrics[f"ap50_{cls.lower().replace(' ', '_')}"]    = float(val.box.ap50[i])
            per_class_metrics[f"ap50_95_{cls.lower().replace(' ', '_')}"] = float(val.box.ap[i])
        except IndexError:
            pass

    wandb.log({
        "final_mAP50":     float(val.box.map50),
        "final_mAP50_95":  float(val.box.map),
        "final_precision": float(val.box.mp),
        "final_recall":    float(val.box.mr),
        "inference_ms":    float(val.speed["inference"]),
        "model_size_mb":   round(os.path.getsize(weights) / 1e6, 2),
        **per_class_metrics
    })

    wandb.finish()
    print(f"✅ {model_name} per-class metrics logged")